# Stage 2 - GEE Farm Data Fetcher

## Cell 1 - Authenticate GEE

In [ ]:
import ee

try:
    ee.Initialize()
    print("GEE ready:", ee.data.getAssetRoots())
except Exception as ex:
    print(f"Initialize failed ({ex.__class__.__name__}), re-authenticating...")
    ee.Authenticate(force=True)
    ee.Initialize()
    print("GEE ready.")

## Cell 2 - Imports

In [ ]:
import sys, os, requests, math
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from datetime import date
from dateutil.relativedelta import relativedelta
from PIL import Image

MODEL_PATH     = "../saved_model/yield_model_v2.joblib"
FEAT_COLS_PATH = "../saved_model/feature_cols_v2.joblib"
model     = joblib.load(MODEL_PATH)
FEAT_COLS = joblib.load(FEAT_COLS_PATH)
print("Model loaded. Features:", FEAT_COLS)

## Cell 3 - Verify each GEE data source (single month test)

In [ ]:
LAT, LON = 33.88, -5.55
TEST_START, TEST_END = "2024-01-01", "2024-02-01"

point = ee.Geometry.Point([LON, LAT])
farm  = point.buffer(500)
wide  = point.buffer(10_000)

# Sentinel-2: pixel math on B3/B4/B8 bands -> indices -> mean over farm area
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(farm).filterDate(TEST_START, TEST_END)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
        .select(["B3","B4","B8"]))
n_s2 = s2.size().getInfo()
ndvi_test = (s2.map(lambda img: img.addBands(
    img.select("B8").subtract(img.select("B4"))
       .divide(img.select("B8").add(img.select("B4")).add(1e-9)).rename("NDVI")))
    .select("NDVI").median()
    .reduceRegion(ee.Reducer.mean(), farm, 10).getInfo())
print(f"Sentinel-2 : {n_s2} images  |  NDVI = {ndvi_test['NDVI']:.4f}")

# MODIS: thermal infrared -> land surface temperature -> convert to Celsius
lst_val = (ee.ImageCollection("MODIS/061/MOD11A1")
    .filterDate(TEST_START, TEST_END).select("LST_Day_1km")
    .mean().multiply(0.02).subtract(273.15)
    .reduceRegion(ee.Reducer.mean(), wide, 1000).getInfo())
print(f"MODIS LST  : {lst_val['LST_Day_1km']:.2f} C")

# CHIRPS: daily rainfall grids summed to monthly total
rain_val = (ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
    .filterDate(TEST_START, TEST_END).select("precipitation")
    .sum().reduceRegion(ee.Reducer.mean(), wide, 5000).getInfo())
print(f"CHIRPS     : {rain_val['precipitation']:.2f} mm")

# SMAP v008: microwave radar soil moisture (m3/m3 -> %)
smap_count = ee.ImageCollection("NASA/SMAP/SPL4SMGP/008").filterDate(TEST_START, TEST_END).size().getInfo()
sm_val = (ee.ImageCollection("NASA/SMAP/SPL4SMGP/008")
    .filterDate(TEST_START, TEST_END).select("sm_surface")
    .mean().multiply(100)
    .reduceRegion(ee.Reducer.mean(), wide, 11000).getInfo())
print(f"SMAP v008  : {smap_count} images  |  soil_moisture = {sm_val['sm_surface']:.2f} %")

## Cell 4 - Fetcher functions

In [ ]:
def _fetch_month(lat, lon, year, month, radius_m=500):
    start  = f"{year}-{month:02d}-01"
    end    = (date(year, month, 1) + relativedelta(months=1)).strftime("%Y-%m-%d")
    pt     = ee.Geometry.Point([lon, lat])
    farm   = pt.buffer(radius_m)
    wide   = pt.buffer(10_000)   # coarser datasets (MODIS 1km, CHIRPS 5km, SMAP 11km)

    # Sentinel-2 spectral indices — standard remote sensing formulas:
    #   NDVI  : Rouse et al. (1974), J. Geophys. Res.
    #   GNDVI : Gitelson et al. (1996), Remote Sens. Environ.
    #   NDWI  : Gao (1996), Remote Sens. Environ.
    #   SAVI  : Huete (1988), Remote Sens. Environ. — L=0.5 default for intermediate cover
    def add_indices(img):
        nir, red, grn = img.select("B8"), img.select("B4"), img.select("B3")
        return img.addBands([
            nir.subtract(red).divide(nir.add(red).add(1e-9)).rename("NDVI"),
            nir.subtract(grn).divide(nir.add(grn).add(1e-9)).rename("GNDVI"),
            grn.subtract(nir).divide(grn.add(nir).add(1e-9)).rename("NDWI"),
            nir.subtract(red).multiply(1.5).divide(nir.add(red).add(0.5)).rename("SAVI"),
        ])

    s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(farm).filterDate(start, end)
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
            .select(["B3","B4","B8"]))

    s2_stats = (s2.map(add_indices).select(["NDVI","GNDVI","NDWI","SAVI"])
                  .median()
                  .reduceRegion(ee.Reducer.mean(), farm, 10, maxPixels=1e9)
                  .getInfo())

    if s2_stats.get("NDVI") is None:
        return None

    # MODIS MOD11A1 Land Surface Temperature
    # Scale factor 0.02, subtract 273.15 K → °C
    # Source: NASA MODIS LST product documentation (Wan, 2014)
    lst = (ee.ImageCollection("MODIS/061/MOD11A1").filterDate(start, end)
             .select("LST_Day_1km").mean().multiply(0.02).subtract(273.15)
             .reduceRegion(ee.Reducer.mean(), wide, 1000, maxPixels=1e9).getInfo())

    # CHIRPS daily precipitation summed to monthly total (mm/month)
    # Source: Funk et al. (2015), Scientific Data, doi:10.1038/sdata.2015.66
    rain = (ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterDate(start, end)
              .select("precipitation").sum()
              .reduceRegion(ee.Reducer.mean(), wide, 5000, maxPixels=1e9).getInfo())

    # NASA SMAP SPL4SMGP v008 surface soil moisture
    # Units: m³/m³ × 100 = volumetric % — standard unit conversion
    # Source: Reichle et al. (2019), SMAP L4 Algorithm Theoretical Basis Document
    smap = (ee.ImageCollection("NASA/SMAP/SPL4SMGP/008").filterDate(start, end)
              .select("sm_surface").mean().multiply(100)
              .reduceRegion(ee.Reducer.mean(), wide, 11000, maxPixels=1e9).getInfo())

    def safe(d, k):
        v = d.get(k)
        return float(v) if v is not None else np.nan

    return {
        "year":          year,
        "month":         month,
        "NDVI":          safe(s2_stats, "NDVI"),
        "GNDVI":         safe(s2_stats, "GNDVI"),
        "NDWI":          safe(s2_stats, "NDWI"),
        "SAVI":          safe(s2_stats, "SAVI"),
        "temperature":   safe(lst,      "LST_Day_1km"),
        "rainfall":      safe(rain,     "precipitation"),
        "soil_moisture": safe(smap,     "sm_surface"),
    }


def fetch_farm_profile(lat, lon, n_months=24, end_year=None, end_month=None, radius_m=500):
    if end_year is None:
        today = date.today()
        end_year, end_month = today.year, today.month

    cur = date(end_year, end_month, 1)
    months = []
    for _ in range(n_months):
        months.append((cur.year, cur.month))
        cur -= relativedelta(months=1)
    months.reverse()

    rows = []
    for i, (y, m) in enumerate(months):
        print(f"  [{i+1:02d}/{n_months}] {y}-{m:02d} ...", end=" ", flush=True)
        row = _fetch_month(lat, lon, y, m, radius_m)
        if row:
            sm_s = f"{row['soil_moisture']:.1f}%" if not np.isnan(row['soil_moisture']) else "N/A"
            print(f"NDVI={row['NDVI']:.3f}  T={row['temperature']:.1f}C  "
                  f"rain={row['rainfall']:.1f}mm  sm={sm_s}")
            rows.append(row)
        else:
            print("no cloud-free image")

    df = pd.DataFrame(rows)
    # interpolate NaN gaps FIRST, then cast to int — order matters
    df = df.interpolate(method="linear", limit_direction="both")
    df["year"]  = df["year"].round().astype(int)
    df["month"] = df["month"].round().astype(int)
    print(f"\nDone: {len(df)} months fetched. NaN counts:\n{df.isna().sum().to_string()}")
    return df

print("Fetcher ready.")

## Cell 5 - Feature engineering

In [ ]:
def build_features(df):
    d = df.copy()
    d["month_sin"]       = np.sin(2 * np.pi * d["month"] / 12)
    d["month_cos"]       = np.cos(2 * np.pi * d["month"] / 12)
    d["veg_stress"]      = (d["NDVI"] - d["GNDVI"]) / (d["NDVI"] + d["GNDVI"] + 1e-9)
    d["water_stress"]    = d["NDVI"] / (d["soil_moisture"] + 1e-9)
    d["ndvi_x_moisture"] = d["NDVI"] * d["soil_moisture"]
    d["thermal_load"]    = np.abs(d["temperature"] - 25.0)
    d["rain_efficiency"] = d["NDVI"] / (d["rainfall"] + 1e-9)
    return d

print("build_features ready.")

## Cell 6 - Fetch 24 months

In [ ]:
# end_year/end_month = last COMPLETE month before today (2026-04-18)
# April 2026 is only 18 days old → use March 2026 as endpoint
# This gives 24 months: April 2024 → March 2026
raw_df = fetch_farm_profile(LAT, LON, n_months=24, end_year=2026, end_month=3)
raw_df

## Cell 7 - Apply Stage 1 model -> yield curve

In [ ]:
feat_df = build_features(raw_df)
feat_df["yield_pred"] = model.predict(feat_df[FEAT_COLS].values)

print(feat_df[["year","month","NDVI","temperature","rainfall","soil_moisture","yield_pred"]].to_string(index=False))
print(f"\nMean={feat_df['yield_pred'].mean():.2f}  Std={feat_df['yield_pred'].std():.2f}  "
      f"Max={feat_df['yield_pred'].max():.2f}  Min={feat_df['yield_pred'].min():.2f}")
print(f"Trend: yr1={feat_df.iloc[:12]['yield_pred'].mean():.2f}  yr2={feat_df.iloc[12:]['yield_pred'].mean():.2f}")

## Cell 8 - Visualize temporal profile

In [ ]:
labels = [f"{int(r['year'])}-{int(r['month']):02d}" for _, r in feat_df.iterrows()]
x = list(range(len(feat_df)))

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
fig.suptitle(f"Farm Profile  ({LAT}, {LON})  Meknes, Morocco", fontsize=13)

for ax, (col, title, color) in zip(axes.flat, [
    ("NDVI",         "NDVI",                     "green"),
    ("yield_pred",   "Predicted Yield (tons/ha)", "steelblue"),
    ("temperature",  "Temperature (C)",           "tomato"),
    ("rainfall",     "Rainfall (mm)",             "royalblue"),
    ("soil_moisture","Soil Moisture (%)",          "saddlebrown"),
    ("veg_stress",   "Vegetation Stress",          "darkorange"),
]):
    ax.plot(x, feat_df[col].values, marker="o", ms=4, color=color)
    ax.set_title(title)
    ax.set_xticks(x[::3])
    ax.set_xticklabels(labels[::3], rotation=45, ha="right", fontsize=7)

plt.tight_layout()
plt.show()

## Cell 9 - Save profile + download 24 RGB satellite images

In [ ]:
os.makedirs("../data/farm_profiles", exist_ok=True)
feat_df.to_csv(f"../data/farm_profiles/farm_{LAT}_{LON}.csv", index=False)
print("Profile saved.")

IMG_DIR  = f"../data/farm_images/{LAT}_{LON}"
os.makedirs(IMG_DIR, exist_ok=True)
vis_area = ee.Geometry.Point([LON, LAT]).buffer(2000).bounds()

def download_rgb(year, month):
    start = f"{year}-{month:02d}-01"
    end   = (date(year, month, 1) + relativedelta(months=1)).strftime("%Y-%m-%d")
    s2    = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
               .filterBounds(vis_area).filterDate(start, end)
               .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
               .select(["B4","B3","B2"]).median())
    url  = s2.getThumbURL({"region": vis_area, "dimensions": 512,
                            "format": "png", "min": 0, "max": 2500, "gamma": 1.4})
    resp = requests.get(url, timeout=30)
    if resp.status_code == 200:
        path = os.path.join(IMG_DIR, f"{year}-{month:02d}.png")
        open(path, "wb").write(resp.content)
        return path
    return None

image_paths = {}
for _, row in feat_df.iterrows():
    y, m  = int(row["year"]), int(row["month"])
    label = f"{y}-{m:02d}"
    path  = download_rgb(y, m)
    image_paths[label] = path
    print(f"  {label}: {'saved' if path else 'no image'}")

print(f"\nDownloaded {sum(v is not None for v in image_paths.values())}/{len(feat_df)} images")

## Cell 10 - Display 24-image grid

In [ ]:
valid = {k: v for k, v in image_paths.items() if v}
cols  = 6
rows  = math.ceil(len(valid) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))

for ax, (label, path) in zip(axes.flat, valid.items()):
    row = feat_df[feat_df.apply(lambda r: f"{int(r['year'])}-{int(r['month']):02d}" == label, axis=1)]
    ndvi_s = f"NDVI={row['NDVI'].values[0]:.3f}" if len(row) else ""
    ax.imshow(Image.open(path))
    ax.set_title(f"{label}\n{ndvi_s}", fontsize=8)
    ax.axis("off")

for ax in axes.flat:
    if not ax.images:
        ax.axis("off")

fig.suptitle(f"Sentinel-2 RGB monthly composites  ({LAT}, {LON})", fontsize=12)
plt.tight_layout()
plt.show()